In [242]:
#!pip install datasets
from IPython.display import display

import pandas as pd
from datasets import load_dataset
import json 
import numpy as np
from scipy.stats import pearsonr
import statistics


pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [106]:
persona = load_dataset("LLM-Digital-Twin/Twin-2K-500", "full_persona")
print("Persona Dataset")
print(persona)
persona_df = persona["data"].to_pandas()


wave_split = load_dataset("LLM-Digital-Twin/Twin-2K-500", "wave_split")

print("Wave Q&A Dateset")
print(wave_split)
wave_df = wave_split["data"].to_pandas()

# Example: Using wave_split for persona creation and evaluation
# train_data = wave_split["data"]["wave1_3_persona_text"]  # or wave1_3_persona_json
# test_questions = wave_split["data"]["wave4_Q_wave4_A"] # you want to remove the "Answers" from all questions
# ground_truth = wave_split["data"]["wave4_Q_wave4_A"]

Persona Dataset
DatasetDict({
    data: Dataset({
        features: ['pid', 'persona_text', 'persona_summary', 'persona_json'],
        num_rows: 2058
    })
})
Wave Q&A Dateset
DatasetDict({
    data: Dataset({
        features: ['pid', 'wave1_3_persona_text', 'wave1_3_persona_json', 'wave4_Q_wave1_3_A', 'wave4_Q_wave4_A'],
        num_rows: 2058
    })
})


### Dataset Statistics

In [107]:
print(persona_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2058 entries, 0 to 2057
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   pid              2058 non-null   object
 1   persona_text     2058 non-null   object
 2   persona_summary  2058 non-null   object
 3   persona_json     2058 non-null   object
dtypes: object(4)
memory usage: 64.4+ KB
None


In [108]:
##### Unique Persons

print(f"Number of uniques persons {persona_df['pid'].nunique()}")


Number of uniques persons 2058


In [109]:
persona_df['persona_summary_wordlen'] = persona_df['persona_summary'].str.split().str.len()
persona_df['persona_text_wordlen'] = persona_df['persona_text'].str.split().str.len()
persona_df['persona_json_wordlen'] = persona_df['persona_json'].str.split().str.len()

In [110]:
print(persona_df.describe())

       persona_summary_wordlen  persona_text_wordlen  persona_json_wordlen
count              2058.000000           2058.000000           2058.000000
mean               2024.820700          23963.736638          23785.397473
std                  94.737785            115.849196            115.146485
min                1782.000000          23469.000000          23253.000000
25%                1963.000000          23890.000000          23712.000000
50%                2014.000000          23957.000000          23778.000000
75%                2077.000000          24025.750000          23847.000000
max                3061.000000          24939.000000          24768.000000


In [111]:
#persona_df.head(1)

##### Statistics - Wave datasets

In [112]:
print(wave_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2058 entries, 0 to 2057
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   pid                   2058 non-null   int64 
 1   wave1_3_persona_text  2058 non-null   object
 2   wave1_3_persona_json  2058 non-null   object
 3   wave4_Q_wave1_3_A     2058 non-null   object
 4   wave4_Q_wave4_A       2058 non-null   object
dtypes: int64(1), object(4)
memory usage: 80.5+ KB
None


In [113]:
print(f"Number of uniques persons {wave_df['pid'].nunique()}")


Number of uniques persons 2058


In [ ]:
wave_df['wave1_3_persona_text_wordlen'] = wave_df['wave1_3_persona_text'].str.split().str.len()
wave_df['wave1_3_persona_json_wordlen'] = wave_df['wave1_3_persona_json'].str.split().str.len()
wave_df['wave4_Q_wave1_3_A_wordlen'] = wave_df['wave4_Q_wave1_3_A'].str.split().str.len()
wave_df['wave4_Q_wave4_A_wordlen'] = wave_df['wave4_Q_wave4_A'].str.split().str.len()

In [115]:
print(wave_df.describe())

               pid  wave1_3_persona_text_wordlen  wave4_Q_wave1_3_A_wordlen  \
count  2058.000000                   2058.000000                2058.000000   
mean   1029.500000                  18102.373178                6230.462099   
std     594.237747                    107.101639                  37.677595   
min       1.000000                  17672.000000                6134.000000   
25%     515.250000                  18034.000000                6199.000000   
50%    1029.500000                  18090.000000                6231.000000   
75%    1543.750000                  18158.000000                6261.000000   
max    2058.000000                  19123.000000                6324.000000   

       wave4_Q_wave4_A_wordlen  
count              2058.000000  
mean               6230.169582  
std                  37.640935  
min                6142.000000  
25%                6199.000000  
50%                6230.500000  
75%                6260.000000  
max                6319.

In [116]:

#wave_df.head(1)
wave_df.columns

Index(['pid', 'wave1_3_persona_text', 'wave1_3_persona_json',
       'wave4_Q_wave1_3_A', 'wave4_Q_wave4_A', 'wave1_3_persona_text_wordlen',
       'wave4_Q_wave1_3_A_wordlen', 'wave4_Q_wave4_A_wordlen'],
      dtype='object')

### Understanding Q & A 

In [117]:
def parse_json_safely(val):
    if isinstance(val, str):
        try:
            return json.loads(val)
        except json.JSONDecodeError:
            return []
    return val if isinstance(val, list) else []

wave_df['wave1_3_persona_json'] = wave_df['wave1_3_persona_json'].apply(parse_json_safely)

In [118]:
wave_df['wave1_3_persona_json'].dtype

dtype('O')

In [212]:
# Step 2: Create a function to flatten the nested structure
def flatten_blocks(row_id, blocks_list):
    records = []
    for block in blocks_list:
        block_name = block.get("BlockName")
        
        # Iterate through the list of questions inside each block
        for q in block.get("Questions", []):
            
            # Safely extract the selected answer text (handles missing answers)
            answers = q.get("Answers", {})
            selected_text = answers.get("SelectedText", None) if isinstance(answers, dict) else None
            
            records.append({
                "participant_id": row_id,
                "block_name": block_name,
                "question_id": q.get("QuestionID"),
                "question_type": q.get("QuestionType"),
                "question_text": q.get("QuestionText"),
                "options": q.get("Options"),
                "selected_answer": selected_text
            })
            
    return records

# Step 3: Apply the function across all rows and combine into a single DataFrame
all_records = []
for index, row in wave_df.iterrows():
    # Pass the DataFrame index (or a specific user ID column) to track participants
    all_records.extend(flatten_blocks(row['pid'], row['wave1_3_persona_json']))

# The final, flattened DataFrame ready for analysis
wave1_3_qa_df = pd.DataFrame(all_records)
print(wave1_3_qa_df.head(1))

   participant_id    block_name question_id question_type  \
0             574  Demographics       QID11            MC   

                                               question_text  \
0  Which part of the United States do you currently live in?   

                                                                                                                                                                                                                                                       options  \
0  [Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME), Midwest (ND, SD, NE, KS, MN, IA, MO, WI, IL, MI, IN, OH), South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL), West (WA, OR, ID, MT, WY, CA, NV, UT, CO, AZ, NM), Pacific (HI, AK)]   

                                                              selected_answer  
0  South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)  


In [213]:
wave1_3_qa_df.shape

(353929, 7)

In [121]:
2058*172

353976

In [138]:
wave1_3_qa_df['participant_id'].value_counts()

participant_id
574     172
1132    172
948     172
1678    172
386     172
       ... 
1916    170
1340    170
1259    170
822     170
983     170
Name: count, Length: 2058, dtype: int64

In [147]:
wave1_3_qa_df['question_type'].unique()

array(['MC', 'DB', 'Matrix', 'TE'], dtype=object)

In [178]:
wave1_3_qa_df[wave1_3_qa_df["question_type"] == 'DB'].sample(3)

,participant_id,block_name,question_id,question_type,question_text,selected_answer
41309,489,Cognitive tests,QID62,DB,"Each of the 10 questions in the next task consists of a word in capital letters, followed by five words as the answer options. Please select the word that is most nearly the SAME in meaning as the word in capital letters (the synonym ). Some of the questions may be challenging. Please complete them to the best of your ability. If you have no idea, please take your best guess. Click &ldquo;Next&rdquo; to go on to the first question.",None
195555,435,Personality,QID262,DB,This completes this section. You will now move on to the next section.,None
163314,1323,Economic preferences,QID243,DB,"The following 6 questions will concern amounts of money you may receive in the future. In particular, these questions will concern amounts of money that you may receive after some delay, for example $5 received in 10 days’ time.",None


In [145]:
#wave1_3_qa_df[wave1_3_qa_df['participant_id']==1].head(50)
wave1_3_qa_df[
    #(wave1_3_qa_df["question_type"] == 'DB') &
    wave1_3_qa_df["block_name"] == "Economic preferences - intro "] 

,participant_id,block_name,question_id,question_type,question_text,selected_answer
70,574,Economic preferences - intro,QID266,DB,"In this final section, you will be asked various questions about your opinions and preferences. In order to help you pace yourself, the ""next"" button will appear only after a few seconds in each question.",None
112,574,Economic preferences - intro,QID242,DB,"In this section, you will be asked various questions about your opinions and preferences. In order to help you pace yourself, the ""next"" button will appear only after a few seconds in each question.",None
242,2001,Economic preferences - intro,QID266,DB,"In this final section, you will be asked various questions about your opinions and preferences. In order to help you pace yourself, the ""next"" button will appear only after a few seconds in each question.",None
284,2001,Economic preferences - intro,QID242,DB,"In this section, you will be asked various questions about your opinions and preferences. In order to help you pace yourself, the ""next"" button will appear only after a few seconds in each question.",None
414,1710,Economic preferences - intro,QID266,DB,"In this final section, you will be asked various questions about your opinions and preferences. In order to help you pace yourself, the ""next"" button will appear only after a few seconds in each question.",None
...,...,...,...,...,...,...
353525,764,Economic preferences - intro,QID242,DB,"In this section, you will be asked various questions about your opinions and preferences. In order to help you pace yourself, the ""next"" button will appear only after a few seconds in each question.",None
353655,1854,Economic preferences - intro,QID266,DB,"In this final section, you will be asked various questions about your opinions and preferences. In order to help you pace yourself, the ""next"" button will appear only after a few seconds in each question.",None
353697,1854,Economic preferences - intro,QID242,DB,"In this section, you will be asked various questions about your opinions and preferences. In order to help you pace yourself, the ""next"" button will appear only after a few seconds in each question.",None
353827,1029,Economic preferences - intro,QID266,DB,"In this final section, you will be asked various questions about your opinions and preferences. In order to help you pace yourself, the ""next"" button will appear only after a few seconds in each question.",None


In [140]:
question_types = (
    wave1_3_qa_df.groupby('block_name').agg(
        unique_question_counts=("question_id", "nunique"),
        question_types=("question_type", lambda x: sorted(x.unique())))
    .reset_index()
    .sort_values(by='unique_question_counts',ascending=False))

print(question_types)

print(f"Total Q Counts {question_types['unique_question_counts'].sum()}")

                      block_name  unique_question_counts        question_types
0               Cognitive tests                       69          [DB, MC, TE]
5                   Personality                       49  [DB, MC, Matrix, TE]
2          Economic preferences                       36  [DB, MC, Matrix, TE]
1                   Demographics                      14                  [MC]
3  Economic preferences - intro                        2                  [DB]
4                   Forward Flow                       1                  [TE]
Total Q Counts 171


##### Q & A Analysis - Wave 4 

##### Part A - Wave 4 and Wave 1_3 overlapping Questions

In [214]:
wave_df['wave4_Q_wave1_3_A'] = wave_df['wave4_Q_wave1_3_A'].apply(parse_json_safely) 


# Step 3: Apply the function across all rows and combine into a single DataFrame
all_records = []
for index, row in wave_df.iterrows():
    # Pass the DataFrame index (or a specific user ID column) to track participants
    all_records.extend(flatten_blocks(row['pid'], row['wave4_Q_wave1_3_A']))

# The final, flattened DataFrame ready for analysis
wave4_13_qa_df = pd.DataFrame(all_records)
#print(wave4_qa_df.head(1))


question_types = (
    wave4_13_qa_df.groupby('block_name').agg(
        unique_question_counts=("question_id", "nunique"),
        question_types=("question_type", lambda x: sorted(x.unique())))
    .reset_index()
    .sort_values(by='unique_question_counts',ascending=False))

print(f"Total Q Counts Wave 4 dataset {question_types['unique_question_counts'].sum()}")
question_types



Total Q Counts Wave 4 dataset 85


,block_name,unique_question_counts,question_types
25,Product Preferences - Pricing,41,"[DB, MC]"
20,Non-experimental heuristics and biases,5,"[MC, Matrix, Slider]"
4,Anchoring - African countries high,2,"[MC, TE]"
5,Anchoring - African countries low,2,"[MC, TE]"
6,Anchoring - redwood high,2,"[MC, TE]"
7,Anchoring - redwood low,2,"[MC, TE]"
28,Proportion dominance 1C,1,[MC]
22,Outcome bias - success,1,[MC]
23,Probability matching vs. maximizing - Problem 1,1,[Matrix]
24,Probability matching vs. maximizing - Problem 2,1,[Matrix]


In [151]:
#wave_df.sample(1)

In [215]:
wave4_13_qa_df.shape

(131712, 7)

In [216]:
wave_df.columns

Index(['pid', 'wave1_3_persona_text', 'wave1_3_persona_json',
       'wave4_Q_wave1_3_A', 'wave4_Q_wave4_A', 'wave1_3_persona_text_wordlen',
       'wave4_Q_wave1_3_A_wordlen', 'wave4_Q_wave4_A_wordlen'],
      dtype='object')

In [154]:
# question_types = (
#     wave4_13_qa_df.groupby('block_name')['question_type']
#     .nunique()
#     .reset_index(name='unique_question_type')
#     .sort_values(by='unique_question_type',ascending=False))
# print(question_types)

#rint(f"Total Q Counts {question_types['unique_question_type'].sum()}")

In [217]:
wave_df['wave4_Q_wave4_A'] = wave_df['wave4_Q_wave4_A'].apply(parse_json_safely) 


# Step 3: Apply the function across all rows and combine into a single DataFrame
all_records = []
for index, row in wave_df.iterrows():
    # Pass the DataFrame index (or a specific user ID column) to track participants
    all_records.extend(flatten_blocks(row['pid'], row['wave4_Q_wave4_A']))

# The final, flattened DataFrame ready for analysis
wave4_4_qa_df = pd.DataFrame(all_records)
print(wave4_4_qa_df.shape)


question_types = (
    wave4_4_qa_df.groupby('block_name').agg(
        unique_question_counts=("question_id", "nunique"),
        question_types=("question_type", lambda x: sorted(x.unique())))
    .reset_index()
    .sort_values(by='unique_question_counts',ascending=False))
question_types
#print(f"Total Question Counts for Wave 4 dataset (No overlap): {question_types['unique_question_counts'].sum()}")



(131712, 7)


,block_name,unique_question_counts,question_types
25,Product Preferences - Pricing,41,"[DB, MC]"
20,Non-experimental heuristics and biases,5,"[MC, Matrix, Slider]"
4,Anchoring - African countries high,2,"[MC, TE]"
5,Anchoring - African countries low,2,"[MC, TE]"
6,Anchoring - redwood high,2,"[MC, TE]"
7,Anchoring - redwood low,2,"[MC, TE]"
28,Proportion dominance 1C,1,[MC]
22,Outcome bias - success,1,[MC]
23,Probability matching vs. maximizing - Problem 1,1,[Matrix]
24,Probability matching vs. maximizing - Problem 2,1,[Matrix]


In [167]:
wave4_13_qa_df.head(3)

,participant_id,block_name,question_id,question_type,question_text,selected_answer
0,574,False consensus,QID287,Matrix,Would you support or oppose...,"[Strongly oppose, Somewhat oppose, Somewhat oppose, Strongly oppose, Somewhat support, Somewhat support, Somewhat support, Somewhat oppose, Somewhat support, Somewhat support]"
1,574,Base-rate 70 engineers,QID156,Slider,"A panel of psychologist have interviewed and administered personality tests to 70 engineers and 30 lawyers, all successful in their respective fields. On the basis of this information, thumbnail descriptions of the 70 engineers and 30 lawyers have been written. Below is one description, chosen at random from the 100 available descriptions. Jack is a 45-year-old man. He is married and has four children. He is generally conservative, careful, and ambitious. He shows no interest in political and social issues and spends most of his free time on his many hobbies which include home carpentry, sailing, and mathematical puzzles. The probability that Jack is one of the 70 engineers in the sample of 100 is ___%. Please indicate the probability on a scale from 0 to 100.",None
2,574,Disease-loss,QID158,MC,"Imagine that the U.S. is preparing for the outbreak of an unusual disease, which is expected to kill 600 people. Two alternative programs to combat the disease have been proposed. Assume that the exact scientific estimate of the consequences of the programs are as follows: If Program A is adopted, 400 people will die. If Program B is adopted, there is 1/3 probability that nobody people will die, and 2/3 probability that 600 people be die. Which of the two programs would you favor?",I slightly favor program A


In [168]:
wave4_4_qa_df.head(3)

,participant_id,block_name,question_id,question_type,question_text,selected_answer
0,574,False consensus,QID287,Matrix,Would you support or oppose...,"[Somewhat oppose, Somewhat support, Neither oppose nor support, Strongly oppose, Somewhat support, Somewhat support, Somewhat support, Somewhat oppose, Somewhat support, Neither oppose nor support]"
1,574,Base-rate 70 engineers,QID156,Slider,"A panel of psychologist have interviewed and administered personality tests to 70 engineers and 30 lawyers, all successful in their respective fields. On the basis of this information, thumbnail descriptions of the 70 engineers and 30 lawyers have been written. Below is one description, chosen at random from the 100 available descriptions. Jack is a 45-year-old man. He is married and has four children. He is generally conservative, careful, and ambitious. He shows no interest in political and social issues and spends most of his free time on his many hobbies which include home carpentry, sailing, and mathematical puzzles. The probability that Jack is one of the 70 engineers in the sample of 100 is ___%. Please indicate the probability on a scale from 0 to 100.",None
2,574,Disease-loss,QID158,MC,"Imagine that the U.S. is preparing for the outbreak of an unusual disease, which is expected to kill 600 people. Two alternative programs to combat the disease have been proposed. Assume that the exact scientific estimate of the consequences of the programs are as follows: If Program A is adopted, 400 people will die. If Program B is adopted, there is 1/3 probability that nobody people will die, and 2/3 probability that 600 people be die. Which of the two programs would you favor?",I slightly favor program B


### Answers Evaluation - Test/Retest 

##### 1. Clean Up - Remove DB Questions 

In [218]:
wave1_3_qa_df[wave1_3_qa_df['question_type']=='DB']['selected_answer'].value_counts()

Series([], Name: count, dtype: int64)

In [219]:
wave4_13_qa_df_cleaned = wave4_13_qa_df[~(wave4_13_qa_df['question_type']=='DB')]
wave4_4_qa_df_cleaned = wave4_4_qa_df[~(wave4_4_qa_df['question_type']=='DB')]
print(wave4_13_qa_df.shape,wave4_4_qa_df.shape)
print(wave4_13_qa_df_cleaned.shape,wave4_4_qa_df_cleaned.shape)
print(wave4_13_qa_df.shape[0]-wave4_13_qa_df_cleaned.shape[0])

(131712, 7) (131712, 7)
(129654, 7) (129654, 7)
2058


##### Create Evaluation Dataset

In [220]:

# 1. Merge the datasets to align past and current answers
eval_df = pd.merge(
    wave4_13_qa_df_cleaned[['participant_id', 'question_id', 'question_type', 'selected_answer']],
    wave4_4_qa_df_cleaned[['participant_id', 'question_id', 'selected_answer']],
    on=['participant_id', 'question_id'],
    suffixes=('_wave13', '_wave4')
)

# # Drop any rows where a human skipped the question in either wave
# original_len = len(eval_df)
# print(original_len)
# eval_df = eval_df.dropna(subset=['selected_answer_wave13', 'selected_answer_wave4'])
# dropped_count = original_len - len(eval_df)
# print(f"Total rows dropped due to NaN: {dropped_count}")
# print(eval_df.shape)

##### 2. Remove Empty responses in either waves


In [221]:

# 1. Replace empty strings or pure whitespace with true NaN
eval_df['selected_answer_wave13'] = eval_df['selected_answer_wave13'].replace(r'^\s*$', np.nan, regex=True)
eval_df['selected_answer_wave4'] = eval_df['selected_answer_wave4'].replace(r'^\s*$', np.nan, regex=True)
# Drop any rows where a human skipped the question in either wave
original_len = len(eval_df)
print(original_len)


# 2. Now count the true missing values
total_skips = eval_df[['selected_answer_wave13', 'selected_answer_wave4']].isna().any(axis=1).sum()
print(f"Total skipped questions (NaN or empty): {total_skips}")

# 3. Finally, execute the drop safely
eval_df = eval_df.dropna(subset=['selected_answer_wave13', 'selected_answer_wave4'])

129654
Total skipped questions (NaN or empty): 10290


In [222]:
eval_df['question_type'].value_counts()

question_type
MC        109074
Matrix     10290
Name: count, dtype: int64

#####  Multiple Choice Question Evaluation - Exact Match

In [223]:
wave4_13_qa_df[wave4_13_qa_df["question_type"] == 'MC'].sample(3)

,participant_id,block_name,question_id,question_type,question_text,options,selected_answer
13815,1208,Product Preferences - Pricing,QID9_32,MC,"Please consider the following product category: Cleaning Supplies. Suppose you are in a grocery store, and you see the following product in that category: ARM & HAMMER Pure Baking Soda, For Baking, Cleaning & Deodorizing, 1 lb Box. The product is priced at: $2.77. Would you or would you not purchase this product?","[Yes, I would purchase the product, No, I would not purchase the product]","Yes, I would purchase the product"
91801,1580,Product Preferences - Pricing,QID9_2,MC,"Please consider the following product category: Bottled Water. Suppose you are in a grocery store, and you see the following product in that category: OZARKA Brand 100% Natural Spring Water, 16.9-ounce plastic bottles (Pack of 35). The product is priced at: $27.94. Would you or would you not purchase this product?","[Yes, I would purchase the product, No, I would not purchase the product]","Yes, I would purchase the product"
66674,1460,Product Preferences - Pricing,QID9_27,MC,"Please consider the following product category: Yogurt - Refrigerated. Suppose you are in a grocery store, and you see the following product in that category: Chobani Non-Fat Greek Yogurt, Vanilla Blended 32 oz, Plastic. The product is priced at: $2.23. Would you or would you not purchase this product?","[Yes, I would purchase the product, No, I would not purchase the product]","Yes, I would purchase the product"


In [224]:
# Isolate Multiple Choice questions
mc_df = eval_df[eval_df['question_type'] == 'MC'].copy()
print(mc_df.shape)
# Calculate Exact Match
mc_df['is_match'] = mc_df['selected_answer_wave13'] == mc_df['selected_answer_wave4']
print(mc_df['is_match'].value_counts())

mc_reliability = mc_df['is_match'].mean()

print(f"Categorical (MC) Exact Match Ceiling: {mc_reliability:.2%}")


(109074, 5)
is_match
True     85156
False    23918
Name: count, dtype: int64
Categorical (MC) Exact Match Ceiling: 78.07%


##### Matrix Type Question Evaluation

In [232]:
wave4_13_qa_df[wave4_13_qa_df["question_type"] == 'Matrix'].sample(3)

,participant_id,block_name,question_id,question_type,question_text,options,selected_answer
36227,1939,Linda -no conjunction,QID159,Matrix,"Linda is 31 years old, single, outspoken, and very bright. She majored in philosophy. As a student, she was deeply concerned with issues of discrimination and social justice, and also participated in anti-nuclear demonstrations. Please complete the statements below.",None,"[Extremely improbable, Extremely improbable, Moderately probable]"
29139,1024,Non-experimental heuristics and biases,QID289,Matrix,"Please rate the following technology or products from ""not at all risky"" to ""extremely risky""",None,"[moderately risky, moderately risky, moderately risky, moderately risky]"
33171,346,Non-experimental heuristics and biases,QID289,Matrix,"Please rate the following technology or products from ""not at all risky"" to ""extremely risky""",None,"[not at all risky, low risk, extremely risky, very risky]"


##### Create Eval dataset

In [245]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import ast

def clean_matrix_cell(val):
    """Safely converts string representations of lists into actual lists."""
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return [val]
    elif isinstance(val, list):
        return val
    return []

# 1. Clean and prepare lists (keeping them unexploded)
m_wave13 = wave4_13_qa_df[wave4_13_qa_df['question_type'] == 'Matrix'].copy()
m_wave4 = wave4_4_qa_df[wave4_4_qa_df['question_type'] == 'Matrix'].copy()

m_wave13['selected_answer'] = m_wave13['selected_answer'].apply(clean_matrix_cell)
m_wave4['selected_answer'] = m_wave4['selected_answer'].apply(clean_matrix_cell)

# 2. Merge directly on participant and question ID
matrix_eval = pd.merge(
    m_wave13[['participant_id', 'question_id', 'selected_answer']],
    m_wave4[['participant_id', 'question_id', 'selected_answer']],
    on=['participant_id', 'question_id'],
    suffixes=('_past', '_current')
)
print(matrix_eval.shape)

(10290, 4)


##### Question Id to possible Answer ranges mapping

In [238]:

# Load the catalog (assuming it is saved as 'question_catalog.json')
with open('../datasets/question_catalog.json', 'r') as f:
    catalog = json.load(f)

# Dictionaries to store our dynamic rules
likert_maps = {}
scale_ranges = {}

for question in catalog:
    if question.get('QuestionType') == 'Matrix' and 'Columns' in question:
        qid = question['QuestionID']
        columns = question['Columns']
        
        # 1. Calculate the range (Number of options - 1)
        # e.g., 5 options = range of 4. 9 options = range of 8.
        scale_ranges[qid] = len(columns) - 1
        
        # 2. Build the text-to-integer mapping for this specific question
        q_map = {}
        for idx, col_text in enumerate(columns):
            # Clean text (lowercase, strip whitespace) to ensure robust matching
            clean_text = str(col_text).strip().lower()
            q_map[clean_text] = idx + 1  # 1-indexed (1, 2, 3...)
            
        likert_maps[qid] = q_map

print(f"Loaded mappings for {len(likert_maps)} matrix questions.")
print(f"Example range for QID29: {scale_ranges.get('QID29')}") # Should output 8
print(f'Scales for QID29')
likert_maps['QID29']

Loaded mappings for 36 matrix questions.
Example range for QID29: 8
Scales for QID29


{'not important to me 1': 1,
 '2': 2,
 '3': 3,
 '4': 4,
 'quite important to me 5': 5,
 '6': 6,
 '7': 7,
 '8': 8,
 'highly important to me 9': 9}

In [243]:
values = list(scale_ranges.values())

print(f"Count  : {len(values)}")
print(f"Mean   : {statistics.mean(values):.2f}")
print(f"Median : {statistics.median(values)}")
print(f"Min    : {min(values)}")
print(f"Max    : {max(values)}")
print(f"Std    : {statistics.stdev(values):.2f}")  # sample standard deviation

Count  : 36
Mean   : 3.11
Median : 4.0
Min    : 1
Max    : 8
Std    : 2.16


In [ ]:
len(scale_ranges)
# 36 

36

#### Map it to our answer response dataset